In [ ]:
import cv2
import torch
import numpy as np
import torch.nn as nn
import csv
import os
import sys

from datetime import datetime
from torchvision import models, transforms
from torchvision.models.resnet import ResNet18_Weights
from PIL import Image

project_root = os.path.abspath("..") 

if project_root not in sys.path:
    sys.path.append(project_root)
from train.model import load_model, transform

device = 'cuda' if torch.cuda.is_available() else 'cpu'

d:\python_envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
filename = 'data.csv'
header = ['Time-stamp', 'True Label', 'Predicted Label']

if not os.path.isfile(filename):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(header)

def append_detection(timestamp, true_label, predicted_label):
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow([timestamp, true_label, predicted_label])


In [ ]:
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_name)
model = load_model(r'../../checkpoint/best_model.pth')

face_cascade = cv2.CascadeClassifier(r'../haarcascade_frontalface_default.xml')

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

cap = cv2.VideoCapture(0)

def draw_probabilities_bar(frame, probabilities):
    start_x = frame.shape[1] - 200 
    start_y = 30  

    for i, prob in enumerate(probabilities):
        label = f"{emotion_labels[i]}: {prob*100:.2f}%" 
        cv2.putText(frame, label, (start_x, start_y + (i * 30)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    return frame

def draw_classes(frame):
    start_x = 0
    start_y = 30

    for i, cls in enumerate(emotion_labels):
        label = f"{i+1}: {cls}"
        cv2.putText(frame, label, (start_x,start_y+(i*30)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
    return frame

def append_result(predicted_emotion, true_emotion):
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        writer.writerow([timestamp, predicted_emotion, true_emotion])

cv2.namedWindow('Emotion Recognition', cv2.WND_PROP_FULLSCREEN)
cv2.setWindowProperty('Emotion Recognition', cv2.WND_PROP_FULLSCREEN, 1)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    frame = cv2.flip(frame, 1)
    frame = draw_classes(frame)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        pil_face = Image.fromarray(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))
        face_tensor = transform(pil_face).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(face_tensor)  
            probabilities = torch.nn.Softmax(dim=1)(outputs)  
            predicted_class = torch.argmax(probabilities, 1) 
            predicted_emotion = emotion_labels[predicted_class.item()] 

        cv2.putText(frame, predicted_emotion, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2) 

        frame = draw_probabilities_bar(frame, probabilities.cpu().numpy().flatten())

    key = cv2.waitKey(1) & 0xFF

    if key == ord('1'):
        append_result(predicted_emotion, 'Angry')
    elif key == ord('2'):
        append_result(predicted_emotion, 'Disgust')
    elif key == ord('3'):
        append_result(predicted_emotion, 'Fear')
    elif key == ord('4'):
        append_result(predicted_emotion, 'Happy')
    elif key == ord('5'):
        append_result(predicted_emotion, 'Neutral')
    elif key == ord('6'):
        append_result(predicted_emotion, 'Sad')
    elif key == ord('7'):
        append_result(predicted_emotion, 'Surprise')
    elif key == ord('q'):
        break

    cv2.imshow('Emotion Recognition', frame)

cap.release()
cv2.destroyAllWindows()
